In [1]:
import sys
sys.path.append('..')
from src.preprocessing.cv_pipeline import preprocess_fold, REDUCED_FEATURES

import pandas as pd
import numpy as np
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

X_ext_train = pd.read_csv('../data/processed/X_ext_train.csv', index_col=0)
X_ext_test = pd.read_csv('../data/processed/X_ext_test.csv', index_col=0)
y_train = pd.read_csv('../data/processed/y_train.csv', index_col=0).squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv', index_col=0).squeeze()

X_reduced_train = X_ext_train[REDUCED_FEATURES]
X_reduced_test = X_ext_test[REDUCED_FEATURES]

disease_names = ['Ankylosing Spondylitis', 'Normal', 'Psoriatic Arthritis',
                  'Reactive Arthritis', 'Rheumatoid Arthritis',
                  "Sjögren's Syndrome", 'Systemic Lupus Erythematosus']

X_reduced_train.shape, X_reduced_test.shape

((9668, 11), (2417, 11))

In [2]:
n_fractional = ((X_reduced_train['ANA'] != 0) & (X_reduced_train['ANA'] != 1)).sum()
print(n_fractional)  # should be ~2995, matching what we confirmed earlier

2995


sd : stacking architecture, base learners : xgb, rf metal learners: either log reg or svc rbf kernel we'll test both and move ahead with whivhever produces better rsuluts

In [3]:
base_learners = [
    ('xgb', XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                           objective='multi:softprob', eval_metric='mlogloss',
                           random_state=42, n_jobs=-1)),
    ('rf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
]

# Version A: LogReg meta-learner
stack_logreg = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    cv=5,
    n_jobs=-1
)
stack_logreg.fit(X_reduced_train, y_train)
y_pred_logreg = stack_logreg.predict(X_reduced_test)

print("=== Stacking with LogReg meta-learner ===")
print(classification_report(y_test, y_pred_logreg, target_names=disease_names, digits=3))

=== Stacking with LogReg meta-learner ===
                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.717     0.595     0.650       425
                      Normal      0.835     0.869     0.852       321
         Psoriatic Arthritis      0.851     0.896     0.873       357
          Reactive Arthritis      0.588     0.913     0.715       103
        Rheumatoid Arthritis      0.856     0.846     0.851       570
          Sjögren's Syndrome      0.879     0.865     0.872       370
Systemic Lupus Erythematosus      1.000     0.985     0.993       271

                    accuracy                          0.834      2417
                   macro avg      0.818     0.853     0.829      2417
                weighted avg      0.836     0.834     0.832      2417



In [5]:
# Version B: SVC meta-learner
stack_svc = StackingClassifier(
    estimators=base_learners,
    final_estimator=SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
    cv=5,
    n_jobs=-1
)
stack_svc.fit(X_reduced_train, y_train)
y_pred_svc = stack_svc.predict(X_reduced_test)

print("=== Stacking with SVC (RBF) meta-learner ===")
print(classification_report(y_test, y_pred_svc, target_names=disease_names, digits=3))

c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


=== Stacking with SVC (RBF) meta-learner ===
                              precision    recall  f1-score   support

      Ankylosing Spondylitis      0.923     0.424     0.581       425
                      Normal      0.973     0.794     0.875       321
         Psoriatic Arthritis      0.826     0.955     0.886       357
          Reactive Arthritis      0.498     0.971     0.658       103
        Rheumatoid Arthritis      0.814     0.921     0.864       570
          Sjögren's Syndrome      0.841     0.989     0.909       370
Systemic Lupus Erythematosus      1.000     0.982     0.991       271

                    accuracy                          0.841      2417
                   macro avg      0.839     0.862     0.823      2417
                weighted avg      0.868     0.841     0.831      2417



In [11]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import pandas as pd

df = pd.read_excel("../data/raw/dataset.xlsx")
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']
for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['Disease'], random_state=42
)
df_train_full = df.loc[train_idx].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

stage3a_results = []
for fold_num, (tr_idx, va_idx) in enumerate(skf.split(df_train_full, df_train_full['Disease']), start=1):
    fold_train_df = df_train_full.iloc[tr_idx]
    fold_val_df = df_train_full.iloc[va_idx]

    _, _, X_ext_tr, X_ext_va, y_tr, y_va = preprocess_fold(fold_train_df, fold_val_df)
    X_red_tr = X_ext_tr[REDUCED_FEATURES]
    X_red_va = X_ext_va[REDUCED_FEATURES]

    stack = StackingClassifier(
        estimators=[
            ('xgb', XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                                   objective='multi:softprob', eval_metric='mlogloss',
                                   random_state=42, n_jobs=1)),
            ('rf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1))
        ],
        final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
        cv=5, n_jobs=1
    )
    stack.fit(X_red_tr, y_tr)
    y_pred = stack.predict(X_red_va)

    stage3a_results.append({
        'fold': fold_num,
        'macro_f1': f1_score(y_va, y_pred, average='macro'),
        'weighted_f1': f1_score(y_va, y_pred, average='weighted'),
    })
    print(f"Fold {fold_num} done")

stage3a_df = pd.DataFrame(stage3a_results)
print(stage3a_df)
print("\nMean macro F1:", stage3a_df['macro_f1'].mean().round(4), "±", stage3a_df['macro_f1'].std().round(4))

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done
   fold  macro_f1  weighted_f1
0     1  0.811274     0.817328
1     2  0.806139     0.820404
2     3  0.814766     0.825492
3     4  0.810956     0.822358
4     5  0.806867     0.819444

Mean macro F1: 0.81 ± 0.0035


stage 3b -> XGBoost (the base learner most likely to benefit from tuning):

max_depth: [3, 5, 7] — controls tree complexity, directly relevant to the AS/RA/PsA overlap problem (deeper trees can carve finer boundaries, but risk overfitting the minority classes)
learning_rate: [0.05, 0.1] — how aggressively it corrects errors each round

Random Forest:

max_depth: [None, 10, 20] — same complexity trade-off
n_estimators: [100, 300] — more trees, generally more stable but slower

Option A — RandomizedSearchCV instead of GridSearchCV. Rather than trying every combination, it randomly samples a fixed number (say, 20) from the space you define. Much faster, usually finds something close to as good as the full grid, and — importantly — lets you directly control your time budget ("try 20 combinations" instead of "try all 108, however long that takes").

In [12]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

base_learners = [
    ('xgb', XGBClassifier(objective='multi:softprob', eval_metric='mlogloss',
                           random_state=42, n_jobs=1)),
    ('rf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1))
]

stack = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    cv=3,   # reduced from 5 -- keeps search tractable, final model gets re-evaluated at cv=5 later
    n_jobs=1
)

param_distributions = {
    'xgb__max_depth': [3, 5, 7],
    'xgb__learning_rate': [0.05, 0.1],
    'rf__max_depth': [None, 10, 20],
    'rf__n_estimators': [100, 300],
    'final_estimator__C': [0.1, 1, 10],
}

search = RandomizedSearchCV(
    estimator=stack,
    param_distributions=param_distributions,
    n_iter=15,
    cv=3,          # outer CV for evaluating each sampled combination
    scoring='f1_macro',
    random_state=42,
    n_jobs=1,      # single-threaded, same memory-safety reasoning as before
    verbose=2      # prints progress so you can see it's actually working, not frozen
)

search.fit(X_reduced_train, y_train)

print("Best params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END final_estimator__C=10, rf__max_depth=None, rf__n_estimators=100, xgb__learning_rate=0.1, xgb__max_depth=7; total time=  11.7s
[CV] END final_estimator__C=10, rf__max_depth=None, rf__n_estimators=100, xgb__learning_rate=0.1, xgb__max_depth=7; total time=  15.2s
[CV] END final_estimator__C=10, rf__max_depth=None, rf__n_estimators=100, xgb__learning_rate=0.1, xgb__max_depth=7; total time=  10.7s
[CV] END final_estimator__C=0.1, rf__max_depth=None, rf__n_estimators=300, xgb__learning_rate=0.1, xgb__max_depth=5; total time=  16.3s
[CV] END final_estimator__C=0.1, rf__max_depth=None, rf__n_estimators=300, xgb__learning_rate=0.1, xgb__max_depth=5; total time=  16.2s
[CV] END final_estimator__C=0.1, rf__max_depth=None, rf__n_estimators=300, xgb__learning_rate=0.1, xgb__max_depth=5; total time=  17.4s
[CV] END final_estimator__C=0.1, rf__max_depth=None, rf__n_estimators=100, xgb__learning_rate=0.1, xgb__max_depth=5; total tim

In [13]:
stack_tuned = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(max_depth=5, learning_rate=0.1, n_estimators=300,
                               objective='multi:softprob', eval_metric='mlogloss',
                               random_state=42, n_jobs=1)),
        ('rf', RandomForestClassifier(max_depth=None, n_estimators=300,
                                       class_weight='balanced', random_state=42, n_jobs=1))
    ],
    final_estimator=LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=42),
    cv=5, n_jobs=1
)

stage3b_results = []
for fold_num, (tr_idx, va_idx) in enumerate(skf.split(df_train_full, df_train_full['Disease']), start=1):
    fold_train_df = df_train_full.iloc[tr_idx]
    fold_val_df = df_train_full.iloc[va_idx]

    _, _, X_ext_tr, X_ext_va, y_tr, y_va = preprocess_fold(fold_train_df, fold_val_df)
    X_red_tr = X_ext_tr[REDUCED_FEATURES]
    X_red_va = X_ext_va[REDUCED_FEATURES]

    stack_tuned.fit(X_red_tr, y_tr)
    y_pred = stack_tuned.predict(X_red_va)

    stage3b_results.append({
        'fold': fold_num,
        'macro_f1': f1_score(y_va, y_pred, average='macro'),
        'weighted_f1': f1_score(y_va, y_pred, average='weighted'),
    })
    print(f"Fold {fold_num} done")

stage3b_df = pd.DataFrame(stage3b_results)
print(stage3b_df)
print("\nMean macro F1:", stage3b_df['macro_f1'].mean().round(4), "±", stage3b_df['macro_f1'].std().round(4))

Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done
   fold  macro_f1  weighted_f1
0     1  0.810606     0.816227
1     2  0.807195     0.822831
2     3  0.814531     0.823547
3     4  0.800051     0.813894
4     5  0.807072     0.821286

Mean macro F1: 0.8079 ± 0.0053


In [14]:
print(stage3a_df['weighted_f1'].mean().round(4))
print(stage3b_df['weighted_f1'].mean().round(4))

0.821
0.8196
